# Release notes
Paths and language were normalized for this source release. Historical training protocols remain documented in `REVIEW-1023.md`; retrain after any protocol corrections.


In [ ]:
from pathlib import Path
import sys
import copy
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "models/DeepLearnModel.py").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the MMHRP-CAMoPE-1023 project.")
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install rdkit==2024.9.5
# !pip install torch_geometric==2.5.3

In [ ]:
import os, sys

# Inspect the current working directory
print("Current working directory:", os.getcwd())

# Inspect the Python module search path
print("Python module search path:")
for p in sys.path:
    print(p)


In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
doc_name = str(PROJECT_ROOT)
# doc_name = r"C:\Users\29868\Desktop\python\Found_tri-modal\autodl\HeckLit-Code-main"
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from torch_geometric.loader import DataLoader
from models.DeepLearnModel import *
import time
from tqdm import tqdm
import datetime
import warnings
warnings.simplefilter('ignore')

In [ ]:
# rs_list = [1,2,3,4,5]
rs_list = [1]
eval_metrics = np.zeros((1+len(rs_list), 6))
columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

In [ ]:
for rs in rs_list:
    # 2.1 Read the data
    data = pd.read_excel("%s/data/Heck/Heck_data_clean.xlsx" % doc_name)
    random_state = rs
    data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)
    vocab_path = "%s/models/HeckLit_vocab.txt" % doc_name

    # 2.2 Construct the reaction list
    rxn_list = df_to_rxn_list(data)
    rxn_RxnSmi = []
    max_len = -1
    for rxn in rxn_list:
        RxnSmi = get_Heck_RxnSmi(rxn)
        max_len = max(max_len, len(RxnSmi))
        RxnSmi = " ".join(smi_tokenizer(RxnSmi))
        rxn_RxnSmi.append(RxnSmi)

    # 2.3 Construct the dataset
    rxn_dataset = []
    smi_inputsize = 128
    for batch in tqdm(range(len(rxn_list))):
        rxn = rxn_list[batch]
        meta = []
        # reactants & products
        reac = rxn.reactants
        prod = rxn.products
        meta.append(smis_to_graph(reac + prod, ChargeContribution=False))
        # solvents, catalysts, reagents
        sols = rxn.solvents
        cats = rxn.cats
        reag = rxn.reagents
        meta.append(smis_to_graph(sols + cats + reag, ChargeContribution=False))
        # RxnSmi
        RxnSmi_vec = RxnSmi_to_tensor(RxnSmi=rxn_RxnSmi[batch], maxlen_=max_len,
                                      victor_size=smi_inputsize, file=vocab_path)
        meta.append(RxnSmi_vec)
        # yield
        meta.append(rxn.rxn_yield / 100)
        rxn_dataset.append(meta)

    # 2.4 Create the output directory
    base_dir = f"{doc_name}/exp/Heck_Benchmark"
    # base_dir = r"C:\Users\29868\Desktop\python\Found_tri-modal\autodl\HeckLit-Code-main\exp\Heck_Benchmark"
    dir_path = os.path.join(base_dir, f"Heck_MMHRP-GCL-MOE_rs=1")
    os.makedirs(dir_path, exist_ok=True)
    report_path = os.path.join(dir_path, "Model_Training_Report.txt")
    f = open(report_path, mode="w")

    # ----------------------
    # 3. Split the dataset
    # ----------------------
    ratio = 0.8
    batch_size = 256
    batch = len(rxn_dataset)
    train_set = rxn_dataset[0: int(ratio * batch)]
    test_set = rxn_dataset[int(ratio * batch) + 1:]
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    # ----------------------
    # 4. Initialize the model
    # ----------------------
    t = 900
    lr = 1e-4
    num_feature = 7
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # device = torch.device("cpu")

    model = MMHRP_GCL_MoE(
        GraphEncoder={
            "NodeFeatNum": num_feature,
            "Channels": [16, 32],
            "Heads": 4
        },
        TextEncoder={
            "SmiFeatNum": smi_inputsize,
            "Heads": 2,
            "BigruChannels": [128, 128],
            "BigruNumlayer": 1
        },
        Decoder={
            "Channels": [384, 128, 64]
        },
        emb_size=128,
        device=device
    )

    opti = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    model = model.to(device)
    criterion = nn.MSELoss()

    # writedown params
    f.write("params:\n")
    f.write(f"random_state={random_state}\n")
    f.write(f"ratio={ratio}\n")
    f.write(f"batch_size={batch_size}\n")
    f.write(f"t={t}\n")
    f.write(f"lr={lr}\n")
    f.write("\nStart training\n")

    # ----------------------
    # 5. Training
    # ----------------------
    best = [0, 0, 0, 0, 0, 0, [], [], []]  # Best-epoch metrics, predictions, and model snapshot
    train_R2, train_RMSE, train_MAE = [], [], []
    test_R2, test_RMSE, test_MAE = [], [], []

    for epoch in tqdm(range(t)):
        global_loss = 0.0
        model.train()
        for data in train_loader:
            x = [i.to(device) for i in data[:-1]]
            y = torch.unsqueeze(data[-1], dim=1).to(device)
            loss = criterion(model(x), y)
            opti.zero_grad()
            loss.backward()
            opti.step()
            global_loss += loss.item()

        # ----------------------
        # Evaluate the training set
        # ----------------------
        model.eval()
        with torch.no_grad():
            pred, true = [], []
            for data in train_loader:
                x = [i.to(device) for i in data[:-1]]
                tr = list(torch.unsqueeze(data[-1], dim=1).detach().cpu().numpy())
                pr = list(model(x).detach().cpu().numpy())
                pred += pr
                true += tr
            train_RMSE.append(RMSE(np.array(pred), np.array(true)))
            train_R2.append(R2(np.array(pred), np.array(true)))
            train_MAE.append(MAE(np.array(true), np.array(pred)))

            # Evaluate the test set
            pred, true = [], []
            for data in test_loader:
                x = [i.to(device) for i in data[:-1]]
                tr = list(torch.unsqueeze(data[-1], dim=1).detach().cpu().numpy())
                pr = list(model(x).detach().cpu().numpy())
                pred += pr
                true += tr
            test_RMSE.append(RMSE(np.array(pred), np.array(true)))
            test_R2.append(R2(np.array(pred), np.array(true)))
            test_MAE.append(MAE(np.array(true), np.array(pred)))

            # Save the best model
            if epoch == 0 or test_R2[-1] > best[3]:
                best = [train_R2[-1], train_RMSE[-1], train_MAE[-1],
                        test_R2[-1], test_RMSE[-1], test_MAE[-1],
                        pred, true, copy.deepcopy(model.state_dict())]

        # Write the report
        f.write(f"Epoch:{epoch+1} loss:{global_loss/batch_size:.6f} "
                f"train_R2:{train_R2[-1]:.3f} test_R2:{test_R2[-1]:.3f} "
                f"train_RMSE:{train_RMSE[-1]:.3f} test_RMSE:{test_RMSE[-1]:.3f} "
                f"train_MAE:{train_MAE[-1]:.3f} test_MAE:{test_MAE[-1]:.3f}\n")

    # ----------------------
    # 6. Record the final evaluation metrics
    # ----------------------
    eval_metrics.loc[str(rs)]["train_R2"] = best[0]
    eval_metrics.loc[str(rs)]["train_RMSE"] = best[1]
    eval_metrics.loc[str(rs)]["train_MAE"] = best[2]
    eval_metrics.loc[str(rs)]["test_R2"] = best[3]
    eval_metrics.loc[str(rs)]["test_RMSE"] = best[4]
    eval_metrics.loc[str(rs)]["test_MAE"] = best[5]

    f.close()

    # ----------------------
    # 7. Save the model
    # ----------------------
    torch.save(best[8], f"{dir_path}/model.pth")

    # ----------------------
    # 8. Visualize the results
    # ----------------------
    import matplotlib.pyplot as plt

    fig = plt.figure(dpi=300, figsize=(10, 5))

    # Training Fig
    plt.subplot(1, 2, 1)
    steps = np.arange(1, t + 1)
    plt.plot(steps, train_R2, color='orange')
    plt.plot(steps, test_R2, color='blue')
    plt.legend(["train set", "test set"], loc="upper left", prop={'size': 8})
    plt.xlabel("Epoch", fontsize=10)
    plt.ylabel("R$^2$", fontsize=10)
    plt.title("R$^2$ during training", fontsize=13)

    # Test set performance
    plt.subplot(1, 2, 2)
    tr = np.array(best[7]).flatten() * 100
    pr = np.array(best[6]).flatten() * 100
    plt.scatter(pr, tr, alpha=0.7, marker=".")
    plt.xlabel("Predicted Yield (%)", fontsize=10)
    plt.ylabel("Observed Yield (%)", fontsize=10)
    x = np.linspace(0, 100, 100)
    y = np.linspace(0, 100, 100)
    plt.plot(x, y, linestyle="--", color="r")
    plt.title("Test set performance", fontsize=13)

    fig.suptitle("MMHRP-GCL-MOE for HeckLit dataset", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{dir_path}/Performance_Figure.png")
    plt.show()

In [ ]:
# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
# eval_metrics.to_csv("%s/exp/Heck_Benchmark/MMHRP-GCL_report_%s.csv" % (doc_name, datetime.datetime.now()))
eval_metrics.to_csv(str(PROJECT_ROOT / 'exp/Heck_Benchmark/Heck_MMHRP-GCL-MOE_rs=1/MMHRP-GCL-MOE_report_rs=1.csv') )#% (doc_name, datetime.datetime.now()
# eval_metrics.to_csv("C:\Users\29868\Desktop\python\Found_tri-modal\autodl\HeckLit-Code-main\exp\Heck_Benchmark/MMHRP-GCL-MOE_report_rs=1.csv" )#% (doc_name, datetime.datetime.now()
print(eval_metrics)